# MERIDA Dataset: Chicago Hyetograph Maps — 3h Duration

Applies the **Chicago (Keifer & Chu) hyetograph** to the MERIDA 3h precipitation
return-value GeoTIFFs, producing peak-intensity maps and hyetograph plots for
each available return period.

**Inputs** (`/home/admin_climatecharted_com/data/MERIDA/RP_hazard_maps/`):
```
pr_3h_MERIDA_2RP_ita.tif
pr_3h_MERIDA_5RP_ita.tif
pr_3h_MERIDA_10RP_ita.tif
pr_3h_MERIDA_25RP_ita.tif
pr_3h_MERIDA_50RP_ita.tif
pr_3h_MERIDA_100RP_ita.tif
pr_3h_MERIDA_200RP_ita.tif
```

**Key differences from the MORE notebook:**
- Input is GeoTIFF (rasterio) instead of NetCDF (xarray)
- Duration is fixed at **3h** only
- All available return periods are processed in one loop
- No pixel-wise IDF fitting needed — scalar `a, b, n` are used
  (or optionally loaded from MORE `idf_param_a.nc` / `idf_param_n.nc`)

## 1. Imports & configuration

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import from_levels_and_colors

try:
    import rasterio
    from rasterio.transform import from_bounds
    from rasterio.crs import CRS
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    raise ImportError('rasterio is required to read GeoTIFF inputs. '
                      'Install with: conda install -c conda-forge rasterio')

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not found – maps will use plain pcolormesh')

print('All imports OK')

In [ ]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────

# Folder containing the MERIDA RP GeoTIFFs
MERIDA_DIR  = '/home/admin_climatecharted_com/data/MERIDA/RP_hazard_maps'

# Output folder
CHICAGO_DIR = '/home/admin_climatecharted_com/data/MERIDA/chicago_3h'
os.makedirs(CHICAGO_DIR, exist_ok=True)

# Duration (fixed for this dataset)
DURATION_H  = 3

# Return periods available in MERIDA (years)
RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 200]

# GeoTIFF filename pattern  — {rp} will be replaced by the RP value
TIF_PATTERN = 'pr_3h_MERIDA_{rp}RP_ita.tif'

# ── Chicago hyetograph parameters ─────────────────────────────────────────────
CHICAGO_R  = 0.35   # peak-position ratio (Italian standard)
CHICAGO_A  = 0.65   # IDF scale  a  [mm/hr]
CHICAGO_B  = 0.10   # IDF offset b  [hr]
CHICAGO_N  = 0.72   # IDF decay  n  [–]
CHICAGO_DT = 5.0    # timestep   [minutes]

# ── (Optional) load pixel-wise a, n from MORE fitting ─────────────────────────
MORE_IDF_DIR = '/home/admin_climatecharted_com/data/MOloch/IDF_results'

# ── Physical depth cap for masking GEV blow-up pixels ─────────────────────────
MAX_PHYSICAL_DEPTH = 500   # mm  for 3h duration

# ── Reference point for single-pixel hyetograph plots ─────────────────────────
POINT_NAME = 'Milan'
POINT_LAT  = 45.46
POINT_LON  = 9.19

# ── Return period to use for the single-point hyetograph plot ─────────────────
PLOT_RP    = 100

# ──────────────────────────────────────────────────────────────────────────────
print(f'Duration     : {DURATION_H}h')
print(f'Return periods: {RETURN_PERIODS}')
print(f'Chicago params: r={CHICAGO_R}, a={CHICAGO_A}, b={CHICAGO_B}, n={CHICAGO_N}')
print(f'Output dir   : {CHICAGO_DIR}')

## 2. Chicago hyetograph helper functions
*(identical to MORE notebook — no changes needed)*

In [ ]:
def create_chicago_hyetograph(
    total_precip_mm: float,
    duration_hours: float,
    dt_minutes: float,
    r: float,
    a: float,
    b: float,
    n: float,
) -> pd.DataFrame:
    """
    Generate a Chicago (Keifer & Chu, 1957) design hyetograph.

    Algorithm: work from the peak outward on each limb using the IDF
    cumulative curve  P(ta) = a * ta / (ta + b)^n  [mm], where ta is
    the reduced time measured from the peak.  Incremental depths on each
    limb are then slotted back around the peak bin so the largest
    increments are nearest the peak.
    """
    dt_hr  = dt_minutes / 60.0
    D      = duration_hours
    n_pre  = int(round(r * D / dt_hr))
    n_post = int(round((1 - r) * D / dt_hr))

    def P(ta):
        return a * ta / (ta + b) ** n

    ta_pre          = np.arange(1, n_pre + 1) * dt_hr
    dP_pre          = np.diff(P(ta_pre), prepend=0.0)
    pre_intensities = dP_pre[::-1] / dt_hr

    ta_post          = np.arange(1, n_post + 1) * dt_hr
    dP_post          = np.diff(P(ta_post), prepend=0.0)
    post_intensities = dP_post / dt_hr

    intensities    = np.concatenate([pre_intensities, post_intensities])
    computed_depth = np.sum(intensities) * dt_hr
    if computed_depth > 0:
        intensities *= total_precip_mm / computed_depth

    times_min = np.arange(n_pre + n_post) * dt_minutes
    return pd.DataFrame({
        'time_min':        times_min,
        'intensity_mm_hr': intensities,
        'depth_mm':        intensities * dt_hr,
    })


def chicago_peak_map(depth_2d, duration_hours, dt_minutes, r, a, b, n):
    """
    Vectorised Chicago peak intensity for a 2-D depth raster.
    Builds a unit-depth template then scales each pixel by its own depth.
    NaN pixels in depth_2d propagate to NaN in the output (ocean masking).
    """
    template = create_chicago_hyetograph(
        total_precip_mm=1.0, duration_hours=duration_hours,
        dt_minutes=dt_minutes, r=r, a=a, b=b, n=n
    )
    shape = template['intensity_mm_hr'].values   # (T,)

    if not np.any(np.isfinite(shape)) or np.nanmax(shape) == 0:
        raise ValueError(
            f'chicago_peak_map: degenerate template — check params: '
            f'a={a}, b={b}, n={n}'
        )

    hyeto_cube = shape[:, None, None] * depth_2d[None, :, :]   # (T, lat, lon)
    peak       = hyeto_cube.max(axis=0)                         # (lat, lon)
    return peak.astype(np.float32), hyeto_cube.astype(np.float32), template['time_min'].values


print('Helper functions defined.')

## 3. Load MERIDA depth rasters from GeoTIFF

In [ ]:
# ── 3.1 Read all return-period GeoTIFFs ───────────────────────────────────────
# Store: depth_data[rp] = np.ndarray (lat, lon) in mm
depth_data = {}
lat_arr = lon_arr = None

for rp in RETURN_PERIODS:
    fname = TIF_PATTERN.format(rp=rp)
    fpath = os.path.join(MERIDA_DIR, fname)

    if not os.path.exists(fpath):
        print(f'  RP{rp:>3d}  MISSING ({fname})')
        continue

    with rasterio.open(fpath) as src:
        arr = src.read(1).astype(np.float32)   # (rows, cols)
        nodata = src.nodata
        transform = src.transform
        height, width = arr.shape

        # Build lat/lon coordinate arrays from the affine transform
        # (pixel-centre coordinates)
        if lat_arr is None:
            col_centres = np.arange(width)  * transform.a + transform.c + transform.a / 2
            row_centres = np.arange(height) * transform.e + transform.f + transform.e / 2
            # rasterio stores rows top-to-bottom (north → south), so row_centres
            # are descending.  We flip to ascending lat for consistency with xarray.
            lon_arr = col_centres.astype(np.float64)
            lat_arr = row_centres[::-1].astype(np.float64)   # ascending

    # Replace nodata with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[arr <= 0] = np.nan

    # rasterio reads north-first; flip to match ascending lat_arr
    arr = arr[::-1, :]

    n_finite = int(np.sum(np.isfinite(arr)))
    print(f'  RP{rp:>3d}  {fname}  '
          f'finite={n_finite}  '
          f'min={np.nanmin(arr):.1f}  p50={np.nanpercentile(arr,50):.1f}  '
          f'max={np.nanmax(arr):.1f} mm')

    depth_data[rp] = arr

print(f'\nGrid: {len(lat_arr)} rows × {len(lon_arr)} cols')
print(f'Lat : [{lat_arr.min():.3f}, {lat_arr.max():.3f}]')
print(f'Lon : [{lon_arr.min():.3f}, {lon_arr.max():.3f}]')

## 3b. (Optional) Load pixel-wise IDF parameters from MORE fitting
If `idf_param_a.nc` / `idf_param_n.nc` exist (produced by the MORE Chicago notebook),
they can be reprojected onto the MERIDA grid and used instead of scalar `a, n`.

In [ ]:
a_path = os.path.join(MORE_IDF_DIR, 'idf_param_a.nc')
n_path = os.path.join(MORE_IDF_DIR, 'idf_param_n.nc')

if os.path.exists(a_path) and os.path.exists(n_path):
    ds_a = xr.open_dataset(a_path)
    ds_n = xr.open_dataset(n_path)

    # Interpolate MORE grid onto MERIDA grid
    A_MAP = ds_a['a'].interp(lat=lat_arr, lon=lon_arr, method='linear').values.astype(np.float32)
    N_MAP = ds_n['n'].interp(lat=lat_arr, lon=lon_arr, method='linear').values.astype(np.float32)
    ds_a.close(); ds_n.close()

    a_med = float(np.nanmedian(A_MAP))
    n_med = float(np.nanmedian(N_MAP))

    if not np.isfinite(a_med) or not np.isfinite(n_med):
        print('WARNING: interpolated A_MAP/N_MAP are all NaN — falling back to scalar params.')
        USE_SPATIAL_PARAMS = False
        a_use = CHICAGO_A
        n_use = CHICAGO_N
    else:
        USE_SPATIAL_PARAMS = True
        a_use = a_med
        n_use = n_med
        print(f'Pixel-wise params loaded and interpolated.')
        print(f'  a: median={a_med:.4f}  (scalar fallback={CHICAGO_A})')
        print(f'  n: median={n_med:.4f}  (scalar fallback={CHICAGO_N})')
else:
    print('MORE parameter rasters not found — using scalar CHICAGO_A, CHICAGO_N.')
    USE_SPATIAL_PARAMS = False
    a_use = CHICAGO_A
    n_use = CHICAGO_N

print(f'\nUsing: a={a_use:.4f}  n={n_use:.4f}  b={CHICAGO_B}  r={CHICAGO_R}')

## 4. Compute Chicago peak-intensity maps for all return periods

In [ ]:
peak_maps   = {}   # {rp: np.ndarray (lat, lon)}
hyeto_cubes = {}   # {rp: np.ndarray (T, lat, lon)}
time_axis   = None

for rp in RETURN_PERIODS:
    if rp not in depth_data:
        print(f'  RP{rp}: depth raster missing — skipping.')
        continue

    depth_2d = depth_data[rp].copy()

    # Mask unphysical pixels
    bad = (~np.isfinite(depth_2d)
           | (depth_2d < 0)
           | (depth_2d > MAX_PHYSICAL_DEPTH))
    n_bad = int(bad.sum())
    if n_bad > 0:
        print(f'  RP{rp:>3d}: masking {n_bad} unphysical pixels '
              f'({n_bad/depth_2d.size*100:.2f}%)')
    depth_2d[bad] = np.nan

    if not np.any(np.isfinite(depth_2d)):
        print(f'  RP{rp:>3d}: all pixels masked — skipping.')
        continue

    print(f'  RP{rp:>3d}: computing Chicago peak map …', end=' ')
    peak, cube, t_min = chicago_peak_map(
        depth_2d,
        duration_hours=DURATION_H,
        dt_minutes=CHICAGO_DT,
        r=CHICAGO_R,
        a=a_use,
        b=CHICAGO_B,
        n=n_use
    )

    peak_maps[rp]   = peak
    hyeto_cubes[rp] = cube
    if time_axis is None:
        time_axis = t_min

    valid = peak[np.isfinite(peak)]
    if valid.size == 0:
        print(f'no finite values — check inputs.')
    else:
        print(f'done.  peak i: min={valid.min():.1f}  '
              f'max={valid.max():.1f}  mean={valid.mean():.1f} mm/hr')

## 5. Single-pixel hyetograph plot

In [ ]:
if PLOT_RP not in depth_data:
    print(f'RP{PLOT_RP} not available — adjust PLOT_RP in Section 1.')
else:
    # Find nearest pixel to the reference point
    lat_idx = int(np.argmin(np.abs(lat_arr - POINT_LAT)))
    lon_idx = int(np.argmin(np.abs(lon_arr - POINT_LON)))
    depth_pt = float(depth_data[PLOT_RP][lat_idx, lon_idx])

    if not np.isfinite(depth_pt):
        print(f'Reference point ({POINT_LAT}, {POINT_LON}) is masked (ocean/nodata).')
        print('Adjust POINT_LAT / POINT_LON in Section 1 to a land pixel.')
    else:
        df_hyet = create_chicago_hyetograph(
            total_precip_mm=depth_pt,
            duration_hours=DURATION_H,
            dt_minutes=CHICAGO_DT,
            r=CHICAGO_R, a=a_use, b=CHICAGO_B, n=n_use
        )

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.bar(
            df_hyet['time_min'], df_hyet['intensity_mm_hr'],
            width=CHICAGO_DT * 0.9, align='edge',
            color='steelblue', alpha=0.8, label=f'\u0394t = {CHICAGO_DT:.0f} min'
        )
        peak_i = df_hyet['intensity_mm_hr'].max()
        peak_t = df_hyet.loc[df_hyet['intensity_mm_hr'].idxmax(), 'time_min']
        ax.axvline(peak_t + CHICAGO_DT / 2, color='red', ls='--', lw=1.2,
                   label=f'Peak {peak_i:.1f} mm/hr @ {peak_t:.0f} min')
        ax.set_xlabel('Time [min]', fontsize=11)
        ax.set_ylabel('Intensity [mm/hr]', fontsize=11)
        ax.set_title(
            f'Chicago hyetograph \u2013 {DURATION_H}h \u2013 RP{PLOT_RP}\n'
            f'{POINT_NAME} ({POINT_LAT}\u00b0N, {POINT_LON}\u00b0E)  |  '
            f'Total depth: {depth_pt:.1f} mm',
            fontsize=10
        )
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        hyet_png = os.path.join(
            CHICAGO_DIR,
            f'chicago_hyetograph_{POINT_NAME.replace(" ","_")}_3h_RP{PLOT_RP}.png'
        )
        fig.savefig(hyet_png, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved \u2192 {hyet_png}')

## 6. Spatial maps of Chicago peak intensity

In [ ]:
def make_precip_cmap():
    levels = [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 130, 160, 200, 250, 300, 400]
    colors = [
        '#ffffff', '#d6e2ff', '#8db2ff', '#626ff7', '#0062ff',
        '#019696', '#01c634', '#63ff01', '#c6ff34', '#ffff02',
        '#ffc601', '#ffa001', '#ff7c00', '#ff1901',
        '#cc0000', '#990033', '#660066',
    ]
    cmap, norm = from_levels_and_colors(levels, colors, extend='max')
    return cmap, norm, levels

PRECIP_CMAP, PRECIP_NORM, PRECIP_LEVELS = make_precip_cmap()


def _make_map(ax, data, lat, lon, title, cmap, norm, label):
    if HAS_CARTOPY:
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()],
                      crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
        ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=':')
        ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
        im = ax.pcolormesh(lon, lat, data, cmap=cmap, norm=norm,
                           transform=ccrs.PlateCarree(), zorder=1)
    else:
        im = ax.imshow(data, origin='lower',
                       extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                       cmap=cmap, norm=norm, aspect='auto')
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(label, fontsize=9)
    ax.set_title(title, fontsize=10, pad=6)
    return im


print('Color scale and map helper ready.')

In [ ]:
# ── 6.1 All-RP overview: one subplot per return period ────────────────────────
rps_available = [rp for rp in RETURN_PERIODS if rp in peak_maps]
n_rp   = len(rps_available)
n_cols = min(n_rp, 4)
n_rows = (n_rp + n_cols - 1) // n_cols

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows),
        subplot_kw={'projection': ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(6 * n_cols, 5 * n_rows),
                              constrained_layout=True)

axes_flat = np.array(axes).flatten()
fig.suptitle(f'MERIDA — Chicago peak intensity [mm/hr] — {DURATION_H}h\n'
             f'r={CHICAGO_R}, a={a_use:.3f}, b={CHICAGO_B}, n={n_use:.3f}',
             fontsize=13)

for k, rp in enumerate(rps_available):
    ax   = axes_flat[k]
    peak = peak_maps[rp].copy()
    peak[peak == 0] = np.nan
    _make_map(ax, peak, lat_arr, lon_arr,
              title=f'RP{rp} yr',
              cmap=PRECIP_CMAP, norm=PRECIP_NORM,
              label='Peak intensity [mm/hr]')

for k in range(n_rp, len(axes_flat)):
    axes_flat[k].set_visible(False)

out_png = os.path.join(CHICAGO_DIR, 'chicago_peak_intensity_3h_all_RP.png')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {out_png}')

In [ ]:
# ── 6.2 Side-by-side: depth vs peak intensity for PLOT_RP ─────────────────────
if PLOT_RP in peak_maps:
    if HAS_CARTOPY:
        fig, (ax1, ax2) = plt.subplots(
            1, 2, figsize=(14, 7),
            subplot_kw={'projection': ccrs.PlateCarree()}
        )
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    depth_plot = depth_data[PLOT_RP].copy()
    depth_plot[depth_plot <= 0] = np.nan
    _make_map(ax1, depth_plot, lat_arr, lon_arr,
              title=f'MERIDA depth — {DURATION_H}h / RP{PLOT_RP}',
              cmap=PRECIP_CMAP, norm=PRECIP_NORM, label='Depth [mm]')

    peak_plot = peak_maps[PLOT_RP].copy()
    peak_plot[peak_plot == 0] = np.nan
    _make_map(ax2, peak_plot, lat_arr, lon_arr,
              title=f'Chicago peak intensity — {DURATION_H}h / RP{PLOT_RP}',
              cmap=PRECIP_CMAP, norm=PRECIP_NORM, label='Peak intensity [mm/hr]')

    plt.suptitle(f'MERIDA {DURATION_H}h — RP{PLOT_RP} — Chicago r={CHICAGO_R}',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    side_png = os.path.join(CHICAGO_DIR,
                             f'chicago_depth_vs_peak_3h_RP{PLOT_RP}.png')
    fig.savefig(side_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {side_png}')

## 7. Export to NetCDF and GeoTIFF

In [ ]:
SAVE_CUBE = False   # set True to also export the full (T, lat, lon) hyetograph cube

def _write_geotiff(array, lat, lon, out_path):
    """Write a 2-D float32 array (lat ascending) to a north-up GeoTIFF."""
    arr = array.astype('float32')
    # Flip back to north-first for GeoTIFF convention
    arr_north = arr[::-1, :]
    dlon = float(lon[1] - lon[0])
    dlat = abs(float(lat[1] - lat[0]))
    west  = float(lon[0])  - dlon / 2
    east  = float(lon[-1]) + dlon / 2
    north = float(lat[-1]) + dlat / 2
    south = float(lat[0])  - dlat / 2
    transform = from_bounds(west, south, east, north,
                            arr_north.shape[1], arr_north.shape[0])
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=arr_north.shape[0], width=arr_north.shape[1],
        count=1, dtype='float32',
        crs=CRS.from_epsg(4326),
        transform=transform,
        compress='lzw', nodata=float('nan'),
    ) as dst:
        dst.write(arr_north, 1)


for rp in rps_available:
    peak  = peak_maps[rp]
    depth = depth_data[rp]
    t_min = time_axis
    coords = {'lat': lat_arr, 'lon': lon_arr}
    enc    = {'dtype': 'float32', 'zlib': True, 'complevel': 4}

    # ── NetCDF ────────────────────────────────────────────────────────────────
    ds_out = xr.Dataset(
        {
            'peak_intensity': xr.DataArray(
                peak, dims=['lat', 'lon'], coords=coords,
                attrs={'units': 'mm/hr',
                       'long_name': f'Chicago peak intensity 3h RP{rp}',
                       'chicago_r': CHICAGO_R, 'idf_a': a_use,
                       'idf_b': CHICAGO_B, 'idf_n': n_use,
                       'dt_min': CHICAGO_DT}
            ),
            'total_depth': xr.DataArray(
                depth, dims=['lat', 'lon'], coords=coords,
                attrs={'units': 'mm',
                       'long_name': f'MERIDA return depth 3h RP{rp}'}
            ),
        },
        attrs={
            'description': 'Chicago (Keifer & Chu) hyetograph on MERIDA 3h depths',
            'return_period_years': rp,
            'duration_hours': DURATION_H,
            'source': 'MERIDA dataset',
        }
    )

    if SAVE_CUBE:
        ds_out['hyetograph'] = xr.DataArray(
            hyeto_cubes[rp],
            dims=['time_min', 'lat', 'lon'],
            coords={'time_min': t_min, **coords},
            attrs={'units': 'mm/hr',
                   'long_name': 'Chicago intensity time series'}
        )

    nc_path = os.path.join(CHICAGO_DIR, f'chicago_3h_RP{rp}.nc')
    ds_out.to_netcdf(nc_path, engine='netcdf4',
                     encoding={'peak_intensity': enc, 'total_depth': enc})

    # ── GeoTIFF ───────────────────────────────────────────────────────────────
    tif_path = os.path.join(CHICAGO_DIR, f'chicago_peak_3h_RP{rp}.tif')
    _write_geotiff(peak, lat_arr, lon_arr, tif_path)

    print(f'  RP{rp:>3d}  NC  \u2192 {nc_path}')
    print(f'         TIF \u2192 {tif_path}')

## 8. Summary statistics

In [ ]:
print(f'MERIDA Chicago hyetograph summary — {DURATION_H}h')
print(f'Parameters: r={CHICAGO_R}, a={a_use:.4f}, b={CHICAGO_B}, n={n_use:.4f}, \u0394t={CHICAGO_DT} min')
print()
col_rp    = 'RP'
col_dmin  = 'Depth min'
col_dmax  = 'Depth max'
col_pmin  = 'Peak i min'
col_pmax  = 'Peak i max'
col_pmean = 'Peak i mean'
header = (f'{col_rp:>6s}{col_dmin:>12s}{col_dmax:>12s}'
          f'{col_pmin:>12s}{col_pmax:>12s}{col_pmean:>13s}')
print(header)
print('\u2500' * len(header))

for rp in rps_available:
    depth = depth_data[rp]
    peak  = peak_maps[rp]
    mask  = np.isfinite(peak) & (peak > 0)
    rp_label = str(rp)
    print(
        f'{rp_label:>6s}'
        f'{np.nanmin(depth):12.1f}'
        f'{np.nanmax(depth):12.1f}'
        f'{peak[mask].min():12.1f}'
        f'{peak[mask].max():12.1f}'
        f'{peak[mask].mean():13.1f}'
    )